# Model Promotion Notebook

This notebook handles the promotion of ML models from one environment to another (e.g., dev → test → prod).

## Process Overview:
1. **Environment Check**: Verify we're not running in development
2. **Parameter Validation**: Extract and validate input parameters
3. **Model Copy**: Copy the champion model to the destination registry
4. **Alias Update**: Set the new model as the active champion

## Prerequisites:
- Source model must exist and have a "champion" alias
- Destination model registry must be accessible
- Appropriate permissions for model registry operations

In [0]:
## 1. Environment Setup and Parameter Extraction

In [ ]:
# Extract parameters from Databricks widgets
try:
    env = dbutils.widgets.get("ENV")
    src_model_name = dbutils.widgets.get("MODEL_NAME")
    dst_model_name = dbutils.widgets.get("DEST_MODEL_NAME")

    print(f"Environment: {env}")
    print(f"Source Model: {src_model_name}")
    print(f"Destination Model: {dst_model_name}")

except Exception as e:
    print(f"Error extracting parameters: {str(e)}")
    dbutils.notebook.exit("Failed to extract required parameters")

In [0]:
# Environment validation - skip promotion in development
if env.lower() == "dev":
    print("⚠️  Model promotion skipped - running in development environment")
    print("   Promotion only allowed in test, uat, or production environments")
    dbutils.notebook.exit("Development environment - no promotion needed")

In [0]:
# Parameter validation
def validate_parameters():
    """Validate that all required parameters are provided and not empty"""
    errors = []

    if not src_model_name or src_model_name.strip() == "":
        errors.append("SOURCE_MODEL_NAME is required and cannot be empty")

    if not dst_model_name or dst_model_name.strip() == "":
        errors.append("DEST_MODEL_NAME is required and cannot be empty")

    if not env or env.strip() == "":
        errors.append("ENV is required and cannot be empty")

    if errors:
        error_msg = "Parameter validation failed:\n" + "\n".join(
            [f"  - {error}" for error in errors]
        )
        print(error_msg)
        dbutils.notebook.exit("Invalid parameters")

    print("✅ Parameter validation successful")


validate_parameters()

In [0]:
## 2. Import Dependencies and Initialize Clients

In [ ]:
# Import required libraries
import os
from typing import Optional

# Databricks Feature Engineering (for future feature store integration)
from databricks.feature_engineering import FeatureLookup, FeatureEngineeringClient

# MLflow for model registry management
import mlflow
from mlflow.tracking.client import MlflowClient

print("✅ Dependencies imported successfully")

In [0]:
# Initialize MLflow client with Unity Catalog
def initialize_mlflow_client():
    """Initialize MLflow client with proper error handling"""
    try:
        registry_uri = "databricks-uc"
        client = MlflowClient(registry_uri=registry_uri)
        mlflow.set_registry_uri(registry_uri)

        print(f"✅ MLflow client initialized with registry: {registry_uri}")
        return client

    except Exception as e:
        error_msg = f"Failed to initialize MLflow client: {str(e)}"
        print(f"❌ {error_msg}")
        dbutils.notebook.exit(error_msg)


# Initialize the client
mlflow_client = initialize_mlflow_client()

In [0]:
## 3. Model Promotion Logic

In [ ]:
# Configuration for model promotion
class ModelPromotionConfig:
    """Configuration class for model promotion settings"""

    # Production model registry path (consider making this configurable)
    PROD_MODEL_REGISTRY = "pru_prod.pac_mlops.fraud_detection"
    CHAMPION_ALIAS = "Champion"
    SOURCE_ALIAS = "champion"

    @staticmethod
    def get_destination_model_name(
        env: str, custom_dst_name: Optional[str] = None
    ) -> str:
        """Get the appropriate destination model name based on environment"""
        if custom_dst_name:
            return custom_dst_name

        # Default behavior - use production registry for non-dev environments
        return ModelPromotionConfig.PROD_MODEL_REGISTRY


# Initialize configuration
config = ModelPromotionConfig()
final_dst_model_name = config.get_destination_model_name(env, dst_model_name)

print(f"📋 Model Promotion Configuration:")
print(f"   Source Model: {src_model_name}@{config.SOURCE_ALIAS}")
print(f"   Destination: {final_dst_model_name}")
print(f"   Target Alias: {config.CHAMPION_ALIAS}")

In [0]:
# Model promotion execution
def promote_model(
    client: MlflowClient, src_name: str, dst_name: str, src_alias: str = "champion"
) -> str:
    """
    Promote a model from source to destination registry

    Args:
        client: MLflow client instance
        src_name: Source model name
        dst_name: Destination model name
        src_alias: Source model alias (default: "champion")

    Returns:
        str: Version of the promoted model
    """
    try:
        # Construct source model URI
        src_model_uri = f"models:/{src_name}@{src_alias}"

        print(f"🚀 Starting model promotion...")
        print(f"   From: {src_model_uri}")
        print(f"   To: {dst_name}")

        # Copy model version
        print(f"📦 Copying model...")
        copied_model_version = client.copy_model_version(src_model_uri, dst_name)

        dest_version = copied_model_version.version
        print(f"✅ Model successfully copied as version {dest_version}")

        return dest_version

    except Exception as e:
        error_msg = f"Model promotion failed: {str(e)}"
        print(f"❌ {error_msg}")
        raise Exception(error_msg)


# Execute model promotion
try:
    promoted_version = promote_model(
        client=mlflow_client,
        src_name=src_model_name,
        dst_name=final_dst_model_name,
        src_alias=config.SOURCE_ALIAS,
    )
except Exception as e:
    dbutils.notebook.exit(str(e))

In [0]:
# Set champion alias for the promoted model
def set_champion_alias(
    client: MlflowClient, model_name: str, version: str, alias: str = "Champion"
) -> None:
    """
    Set the champion alias for a specific model version

    Args:
        client: MLflow client instance
        model_name: Name of the model
        version: Version to set as champion
        alias: Alias to set (default: "Champion")
    """
    try:
        print(f"👑 Setting {alias} alias...")

        client.set_registered_model_alias(name=model_name, alias=alias, version=version)

        print(f"✅ Model {model_name} version {version} successfully set as {alias}")

    except Exception as e:
        error_msg = f"Failed to set champion alias: {str(e)}"
        print(f"❌ {error_msg}")
        raise Exception(error_msg)


# Set the champion alias
try:
    set_champion_alias(
        client=mlflow_client,
        model_name=final_dst_model_name,
        version=promoted_version,
        alias=config.CHAMPION_ALIAS,
    )
except Exception as e:
    dbutils.notebook.exit(str(e))

In [0]:
## 4. Promotion Summary

In [ ]:
# Generate promotion summary
print("=" * 60)
print("🎉 MODEL PROMOTION COMPLETED SUCCESSFULLY!")
print("=" * 60)
print(f"📊 Promotion Summary:")
print(f"   Environment: {env.upper()}")
print(f"   Source Model: {src_model_name}")
print(f"   Destination Model: {final_dst_model_name}")
print(f"   Promoted Version: {promoted_version}")
print(f"   Champion Alias: {config.CHAMPION_ALIAS}")
print("=" * 60)
print(
    f"✅ Model {final_dst_model_name} v{promoted_version} is now active for inference!"
)

# Return success status for downstream processes
dbutils.notebook.exit(
    {
        "status": "success",
        "promoted_model": final_dst_model_name,
        "promoted_version": promoted_version,
        "environment": env,
    }
)